In [0]:
# ======================================================================
# NOTEBOOK 01:DATA INGESTION AND EXPLORATION
# Project: Credit Compass - Loan Lifecycle & Risk intelligence Platform
print("="*80)
print("CREDIT COMPASS: Loan Lifecycle & Risk Intelligence Platform")
print("Notebook 01: Data Ingestion & Exploration")
print("="*80)

# Import required libraries
from pyspark.sql import SparkSession 
from pyspark.sql.functions import *
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import *
import warnings
import pandas as pd
import builtins
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print()

In [0]:
# ==================================================================
# STEP 1: Load Accepted Loans Data
# ==================================================================

print("STEP 1: Loading Accepted Loans Data")
print("="*80)

# File path in Databricks Volume
accepted_path = "/Volumes/workspace/lending_club/lending_club_data/accepted_2007_to_2018Q4.csv"

try:
    #Load the CSV file
    accepted_df = spark.read.csv(accepted_path,header=True,inferSchema=True)

    # Display basic info
    row_count = accepted_df.count()
    col_count = len(accepted_df.columns)

    print("Accepted loans loaded successfully")
    print(f"Total rows: {row_count:,}")
    print(f"Total columns: {col_count}")
    print()

    print("Display random 50 rows:")
    display(accepted_df.orderBy(rand()).limit(50)) 

except Exception as e:
    print(f"Error loading accepted loans data:")
    print(f" {str(e)}")
    print()
    print("Update the 'accepted_path' with actual file path")


In [0]:
# ==================================================================
#  Load Rejected Loans Data Files 
# ==================================================================

print("STEP 2: Loading Rejected Loans Data")
print("="*80)

rejected_path = "/Volumes/workspace/lending_club/lending_club_data/rejected_2007_to_2018Q4.csv"

try:
    # Load the CSV file
    rejected_df = spark.read.csv(rejected_path,header= True,inferSchema= True,multiLine=True,
    escape='"')

    #Display basic info
    row_count = rejected_df.count()
    col_count = len(rejected_df.columns)

    print("Rejected Loans data successfully loaded")
    print(f"Total rows:{row_count}")
    print(f"Total column:{col_count}")

    print("Display random 50 rows")
    display(rejected_df.orderBy(rand()).limit(50))

except Exception as e:
    print(f"Error loading rejected data")
    print(f"Error: str{e}")
    print()
    print("Update the 'rejected_path' with actual file path")

In [0]:
print("Date range in rejected data:")
rejected_df.select(
    min('Application Date').alias('earliest'),
    max('Application Date').alias('latest')
).show()

# Check a sample to confirm rows look valid
print("Sample rows:")
rejected_df.show(5, truncate=False)

In [0]:
# ================================================
# STEP 3: Schema Exploration - Accepted Loans Data
# ================================================

print("STEP 3: Schema Exploration - Aceepted Loand Data")
print("="*80)

# Display Schema 
print("Columns Names and Data Types")
print()
accepted_df.printSchema()

print()
print("Data Types Summary")
dtypes_list = accepted_df.dtypes
print(dtypes_list)

string_cols = [name for name,dtype in dtypes_list if dtype=='string']
double_cols = [name for name,dtype in dtypes_list if dtype=='double']
int_cols = [name for name,dtype in dtypes_list if 'int' in dtype.lower()]

print(f"String columns:{len(string_cols)}")
print(f"Double columns:{len(double_cols)}")
print(f"Integer columns:{len(int_cols)}")








In [0]:
# =================================================================
# STEP 4:Schema Exploration - Rejected Loans
# =================================================================

print("STEP 4:Schema Exploration - Rejected Loans")
print("="*80)

#Display Schema
print("Columns Names and Data Types")
print()
rejected_df.printSchema()

print()
print("Data Types Summary")
dtypes_list = rejected_df.dtypes
print(dtypes_list)

print()
print("Comparison:")
print(f"  Accepted loans columns: {len(accepted_df.columns)}")
print(f"  Rejected loans columns: {len(rejected_df.columns)}")
print()
print("⚠️ Note: Rejected loans typically have fewer columns (application data only)")

In [0]:
# ===================================================================
# STEP 5: DATA QUALITY CHECK - NULL VALUES (ACCEPTED LOANS)
# ===================================================================

print(" STEP 5: Data Quality Check - NUll  Values (Accepted Loans)")
print("="*80)

#total rows
total_rows = accepted_df.count()

# ===============================================================
# 1. Compute null counts for all columns in one pass
# ===============================================================
null_counts_row = accepted_df.select([
    sum(col(column).isNull().cast("int")).alias(column)
    for column in accepted_df.columns]).collect()[0]

# ==========================================================
# 2.Null percentage for each column
# ==========================================================
null_counts = accepted_df.select([
    round(sum(col(c).isNull().cast("int"))/total_rows * 100,2).alias(c)   
    for c in accepted_df.columns])

print("Null percentage for each column")
display(null_counts)

# ==========================================================
# 3. long format:column,null count,mull percentage
# ==========================================================
null_stats = [
    (c,int(null_counts_row[c]),builtins.round(null_counts_row[c]/total_rows*100,2))
    for c in accepted_df.columns
]

schema = StructType([
    StructField('column',StringType(),True),
    StructField('null_count',IntegerType(),True),
    StructField("null_percentage",DoubleType(),True)
])

null_df = spark.createDataFrame(null_stats,schema)

print("Detailed null statistics:")
display(null_df)


# =================================
# 4.Show columns with >50% nulls
# =================================
print("Columns with >50% null values:")
high_null_cols = (
    null_df
    .filter(col("null_percentage")>50)
    .orderBy(col("null_percentage").desc())

)

high_null_count = high_null_cols.count()
print(f"Total: {high_null_count} columns")
display(high_null_cols)

# ===================================
# 5.Columns with 0% nulls
# ===================================
print("Columns with 0% nulls(perfect quality)")
perfect_cols = (
    null_df
    .filter(col("null_percentage")==0)
    .orderBy(col("column"))
)

perfect_count = perfect_cols.count()
print(f"Total: {perfect_count} columns")
display(perfect_cols)

# -------------------------------------------------------------------
# 6. Final note
# -------------------------------------------------------------------
print(" Columns with >50% nulls will be dropped in Notebook 02")

In [0]:
# ===================================================
# STEP 6: Statistical Summary - KEY FINANCIAL COLUMNS
# ===================================================

print("STEP 6: Statistical Summary - Key Financial Columns")
print("-"* 80)

# Key financial columns to analyse
key_columns = ['loan_amnt','funded_amnt','funded_amnt_inv','int_rate',
               'installment','annual_inc','dti','open_acc','total_acc']

# Check which columns exist
existing_key_columns = [col for col in key_columns if col in accepted_df.columns]

if len(existing_key_columns)>0:
    print(f"Analysing columns:{existing_key_columns}")
    print()

    #Display statistical summary
    display(accepted_df.select(existing_key_columns).describe())
else:
    print("None of  the expected financial columns found")
    print("Available columns")
    print(accepted_df.columns[:20])


###  NOTE:
Statistical summary reveals data quality issues where numeric columns contain string values,
leading to incorrect min/max outputs. This indicates schema inference limitations.
These columns will be cleaned and explicitly cast to numeric types in the next stage.

In [0]:
# ===========================================================
# STEP 7: Loan Status Distribution
# ===========================================================

print("STEP 7: Loan Status Distribution")
print("-"* 80)

if 'loan_status' in accepted_df.columns:
    print("Loan Status Breakdown")
    print()

    status_df = (
        accepted_df.groupBy(col('loan_status'))
                   .count()
                   .orderBy(col('count').desc())
    )
    status_df.show(truncate=False)
else:
    print("Loan_status column not found")

In [0]:
# =========================================
# STEP 8: Grade Distribution (Risk Bands)
# =========================================

print("Step 8:Grade Distribution ( Risk Bands)")
print("="*80)

# Check if grade column exists
if 'grade' in accepted_df.columns:
    print("Risk grade breakdown")
    print()

    display(accepted_df.groupBy(col('grade')).count().orderBy('grade'))

    print()
    print("Grades A-G represent risk levels (A = lowest risk, G = highest risk)")
else:
    print("grade column not found")


In [0]:
# =========================================
# Step 9: Time Period Analysis
# =========================================

print("Step 9: Time period Analysis")
print("="*80)

# Check if issue_d column exists
if 'issue_d' in accepted_df.columns:
    
    # Convert issue_d to date (format: MMM-yyyy)
    df_with_date = (
        accepted_df.withColumn(
            'issue_date',
            try_to_date(col('issue_d'),'MMM-yyyy')
        ).filter(
        col('issue_date').isNotNull()  # Skip rows with invalid dates
    ).withColumn(
            'issue_year',
            year(col('issue_date'))
        )
    )

    print('Loan Volume by year:')
    print()

    yearly_df = df_with_date.groupBy(col('issue_year')).agg(
        count('*').alias('loan_count')
    ).orderBy(col('issue_year'))

    display(yearly_df)

else:
    print("issue_d column not found")
     


### Step 9 Observations: Loan Volume by Year (2007–2018)

**Dataset spans 12 years** from 2007 to 2018, covering Lending Club's full operating history.

**Key findings:**
- **Early stage (2007–2010):** Very low volume — only 603 loans in 2007, reflecting Lending Club's startup phase and limited market reach
- **Growth phase (2011–2014):** Volume roughly doubled each year, scaling from 21,721 to 235,629 loans as the platform matured
- **Peak period (2015–2018):** All four years exceeded 400K loans, with 2018 reaching the highest single-year volume of 495,242

**Decision — filter to 2015–2018 for all downstream analysis:**
- Highest data volume and richest feature completeness
- Avoids early-year startup noise and 2008 financial crisis distortions
- These 4 years represent the most relevant and reliable period for credit risk analysis

In [0]:
# ===============================================
#  Step 10: Duplicate Check
# ===============================================

print("Step 10: Duplicate check")
print("="* 80)

total_rows = accepted_df.count()
distinct_rows = accepted_df.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicates}")

if duplicates == 0:
    print("No duplicate rows found")
else:
    print(f"{duplicates} duplicate rows detected")
    print("Will be removed in Notebook 02")
      
# Check if ID column exists for unique identifier check
if 'id' in accepted_df.columns:
    print()
    print("Checking loan ID uniqueness:")
    total_ids = accepted_df.select('id').count()
    unique_ids = accepted_df.select('id').distinct().count()
    duplicate_ids = total_ids - unique_ids

    print(f"  Total loan IDs: {total_ids:,}")
    print(f"  Unique loan IDs: {unique_ids:,}")
    print(f"  Duplicate IDs: {duplicate_ids:,}")


In [0]:
# =======================================
# Step 11: Sample data preview
# =======================================

print("Step 11: Sample data preview")
print("="* 80)

# Select key columns for preview

preview_cols = ['id','loan_amnt','funded_amnt','int_rate','grade',
                'sub_grade','emp_length','annual_inc','loan_status',
                'purpose','dti','issue_d']

# Filter to only existing columns
existing_preview_cols = [col for col in preview_cols if col in accepted_df.columns]

print(f"Preview columns {(len(existing_preview_cols))}: {existing_preview_cols}")
print()

# Display sample
print("Sample loan records:")
accepted_df.select(existing_preview_cols).show(10)
      


In [0]:
# ============================================================================
# STEP 12: REJECTED LOANS - QUICK ANALYSIS
# ============================================================================

print("STEP 12: Rejected Loans - Quick Analysis")
print("-" * 80)

print("Rejected Loans Column Names:")
print(rejected_df.columns)
print()

print("Sample rejected applications:")
rejected_df.show(5, truncate=False)


### Step 12 Observations: Rejected Loans Data

**Rejected loans contain application data only** — no performance history (no loan status, payments, or default information).

This is expected. Rejected applicants never received a loan, so there is nothing to track beyond their initial application details.

**Usage in this project:**
- Rejected data will be used in **Notebook 04** for funnel analysis
- Will help identify what separates accepted vs rejected applicants
- Key comparison columns: DTI, credit score, employment length, loan purpose

## Notebook 01: Data Ingestion & Exploration — Completed ✓

---

### Summary of Findings

| Item | Detail |
|------|--------|
| Accepted loans | 2,260,701 rows, 151 columns |
| Rejected loans | 27,648,741 rows, 9 columns |
| Overall rejection rate | ~92% — only 1 in 12 applications approved |
| Columns with >50% nulls | 44 columns flagged for dropping |
| Columns with 0% nulls | 41 columns with perfect data quality |
| Duplicate rows | 0 full duplicates — All 2,260,701 loan IDs are unique |
| Schema explored | String, Double, Integer types identified |
| Statistical summary | Completed on 9 key financial columns |

---

### Key Insights
- Dataset spans **2007–2018** — full 11+ years of lending history
- **2015–2018** shows highest volume and best data quality — selected for analysis
- **Multiple risk grades (A–G)** available for borrower segmentation
- **Loan status** column captures full performance history (Paid / Current / Default)
- **92% rejection rate** sets up a strong accepted vs rejected comparison in Notebook 04

---

### Next Steps — Notebook 02: Data Cleaning & Filtering
- Filter dataset to **2015–2018** period only
- Drop all columns with **>50% null values** (44 columns)
- Impute remaining nulls (median for DTI, annual income, employment length)
- Fix data types — strip `%` from interest rate, parse date columns
- Save cleaned dataset as **Delta Lake table**